<a href="https://colab.research.google.com/github/JustinRSK/2025_ML_EES/blob/main/ML_Final_project/ML_Final_Project_GC_Justin_Knight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 1. PACKAGES
# ============================================================

# Rasterio may not be installed by default in Google Colab.
%pip install -q rasterio

from pathlib import Path
import subprocess
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
from sklearn.model_selection import LeaveOneGroupOut

print("Packages imported successfully.")

Packages imported successfully.


## Data

Four raster datasets are used.

### Predictors
- `GC_predictors_2011.tif`
- `GC_predictors_2025.tif`

Each contains 12 predictors at 30 m resolution:

1. Blue reflectance
2. Green reflectance
3. Red reflectance
4. Near-infrared reflectance
5. SWIR1 reflectance
6. SWIR2 reflectance
7. NDVI
8. NDMI
9. MNDWI
10. NBR
11. Elevation
12. Slope

### Labels
- `vegetation_class_2011.tif`
- `reserve_id.tif`

The vegetation raster contains six broad land-cover / vegetation classes.

The reserve raster identifies spatially independent reserve groups used for
cross-validation.


In [4]:
# ============================================================
# 2. PROJECT AND DATA PATHS
# ============================================================

REPO_URL = "https://github.com/JustinRSK/2025_ML_EES.git"

# Detect whether the notebook is running in Google Colab.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:

    REPO_DIR = Path("/content/2025_ML_EES")

    if not REPO_DIR.exists():
        print("Cloning GitHub repository...")
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print("Repository already present in Colab.")

    PROJECT_DIR = REPO_DIR / "ML_Final_project"

else:
    # Allows the notebook to also run locally.
    cwd = Path.cwd()

    if cwd.name == "ML_Final_project":
        PROJECT_DIR = cwd
    elif (cwd / "ML_Final_project").exists():
        PROJECT_DIR = cwd / "ML_Final_project"
    else:
        PROJECT_DIR = cwd


# ------------------------------------------------------------
# Required input files
# ------------------------------------------------------------

required_files = [
    "GC_predictors_2011.tif",
    "GC_predictors_2025.tif",
    "vegetation_class_2011.tif",
    "reserve_id.tif"
]


# First look inside ML_Final_project/data.
# If the files are not there, look directly in ML_Final_project.

possible_data_dirs = [
    PROJECT_DIR / "data",
    PROJECT_DIR
]

DATA_DIR = None

for candidate in possible_data_dirs:

    if all((candidate / f).exists() for f in required_files):
        DATA_DIR = candidate
        break


if DATA_DIR is None:
    raise FileNotFoundError(
        "The four required TIFF files could not be found.\n"
        "Place them either in ML_Final_project/data/ "
        "or directly in ML_Final_project/."
    )


OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = PROJECT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)


print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

print("\nInput files:")
for f in required_files:
    print(" ✓", DATA_DIR / f)

Cloning GitHub repository...
Project directory: /content/2025_ML_EES/ML_Final_project
Data directory: /content/2025_ML_EES/ML_Final_project
Output directory: /content/2025_ML_EES/ML_Final_project/outputs

Input files:
 ✓ /content/2025_ML_EES/ML_Final_project/GC_predictors_2011.tif
 ✓ /content/2025_ML_EES/ML_Final_project/GC_predictors_2025.tif
 ✓ /content/2025_ML_EES/ML_Final_project/vegetation_class_2011.tif
 ✓ /content/2025_ML_EES/ML_Final_project/reserve_id.tif


In [5]:
# ============================================================
# 3. MODEL CONFIGURATION
# ============================================================

RANDOM_STATE = 42


# ------------------------------------------------------------
# Predictor names
# ------------------------------------------------------------

FEATURE_NAMES = [
    "blue",
    "green",
    "red",
    "nir",
    "swir1",
    "swir2",
    "NDVI",
    "NDMI",
    "MNDWI",
    "NBR",
    "elevation",
    "slope"
]


# ------------------------------------------------------------
# Vegetation class names
# ------------------------------------------------------------

CLASS_NAMES = {
    1: "Eaux libres",
    2: "Rivages et lieux humides",
    3: "Pelouses et prairies",
    4: "Forêts",
    5: "Plantations/champs/cultures",
    6: "Milieux construits"
}


# ------------------------------------------------------------
# Reserve names
# ------------------------------------------------------------

RESERVE_NAMES = {
    1: "Grèves de Cheseaux",
    2: "Baie d'Yvonand",
    3: "Cheyres",
    4: "Grèves de la Corbière",
    5: "Grèves d'Ostende",
    6: "Grèves de la Motte",
    7: "Cudrefin",
    8: "Fanel neuchâtelois",
    9: "Vernes et Tuileries"
}


# ------------------------------------------------------------
# Random Forest parameters
# ------------------------------------------------------------
#
# The model is deliberately kept relatively conservative.
#
# class_weight="balanced_subsample" reduces the effect of
# class imbalance.
#
# min_samples_leaf=2 adds a small amount of regularisation.
#
# The spatial cross-validation below is the main evaluation
# procedure; training accuracy is NOT used as validation.
# ------------------------------------------------------------

RF_PARAMS = {
    "n_estimators": 500,
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "class_weight": "balanced_subsample",
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# Number of random permutations per feature for PFI.
PFI_N_REPEATS = 10

# To keep PFI computationally reasonable, very large held-out
# reserves can be randomly reduced to this number of test pixels.
# Importantly, these pixels still come ONLY from the held-out
# test reserve.
PFI_MAX_SAMPLES = 15000

# Threshold used only for interpreting candidate 2011–2025 change.
CHANGE_CONFIDENCE_THRESHOLD = 0.70

# Each pixel is 30 x 30 m = 900 m² = 0.09 ha.
PIXEL_AREA_HA = 30 * 30 / 10000

print("Configuration loaded.")

Configuration loaded.


In [6]:
# ============================================================
# 4. LOAD RASTERS
# ============================================================

p2011_path = DATA_DIR / "GC_predictors_2011.tif"
p2025_path = DATA_DIR / "GC_predictors_2025.tif"
class_path = DATA_DIR / "vegetation_class_2011.tif"
reserve_path = DATA_DIR / "reserve_id.tif"


def read_predictor_stack(path):
    """
    Read a multiband raster.

    Masked / NoData pixels are converted to NaN so that they
    can easily be excluded from machine-learning operations.
    """

    with rasterio.open(path) as src:

        data = (
            src.read(masked=True)
            .filled(np.nan)
            .astype(np.float32)
        )

        metadata = {
            "crs": src.crs,
            "transform": src.transform,
            "width": src.width,
            "height": src.height,
            "count": src.count,
            "profile": src.profile.copy(),
            "descriptions": src.descriptions
        }

    return data, metadata


def read_label_raster(path):
    """
    Read a single-band categorical raster.

    NoData pixels are replaced by 0.
    """

    with rasterio.open(path) as src:

        data = (
            src.read(1, masked=True)
            .filled(0)
            .astype(np.int16)
        )

        metadata = {
            "crs": src.crs,
            "transform": src.transform,
            "width": src.width,
            "height": src.height,
            "profile": src.profile.copy()
        }

    return data, metadata


predictors_2011, meta_2011 = read_predictor_stack(p2011_path)
predictors_2025, meta_2025 = read_predictor_stack(p2025_path)

vegetation, meta_vegetation = read_label_raster(class_path)
reserve, meta_reserve = read_label_raster(reserve_path)


print("2011 predictor shape:", predictors_2011.shape)
print("2025 predictor shape:", predictors_2025.shape)
print("Vegetation shape:", vegetation.shape)
print("Reserve shape:", reserve.shape)

2011 predictor shape: (12, 832, 1084)
2025 predictor shape: (12, 832, 1084)
Vegetation shape: (832, 1084)
Reserve shape: (832, 1084)


In [7]:
# ============================================================
# 5. VERIFY THAT ALL RASTERS ARE PERFECTLY ALIGNED
# ============================================================

assert predictors_2011.shape[0] == len(FEATURE_NAMES), (
    "Unexpected number of bands in the 2011 predictor raster."
)

assert predictors_2025.shape[0] == len(FEATURE_NAMES), (
    "Unexpected number of bands in the 2025 predictor raster."
)

assert predictors_2011.shape == predictors_2025.shape, (
    "2011 and 2025 predictor stacks have different dimensions."
)

assert predictors_2011.shape[1:] == vegetation.shape, (
    "Vegetation raster is not aligned with predictors."
)

assert predictors_2011.shape[1:] == reserve.shape, (
    "Reserve raster is not aligned with predictors."
)

assert meta_2011["crs"] == meta_2025["crs"]
assert meta_2011["crs"] == meta_vegetation["crs"]
assert meta_2011["crs"] == meta_reserve["crs"]

assert meta_2011["transform"] == meta_2025["transform"]
assert meta_2011["transform"] == meta_vegetation["transform"]
assert meta_2011["transform"] == meta_reserve["transform"]


print("✓ All raster grids are aligned.")
print("CRS:", meta_2011["crs"])
print(
    "Raster dimensions:",
    meta_2011["width"],
    "x",
    meta_2011["height"]
)

print("\nGeoTIFF band descriptions:")
print(meta_2011["descriptions"])

✓ All raster grids are aligned.
CRS: EPSG:2056
Raster dimensions: 1084 x 832

GeoTIFF band descriptions:
('blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'NDVI', 'NDMI', 'MNDWI', 'NBR', 'elevation', 'slope')


## Training sample construction

A pixel is included in the training dataset only when:

1. all 12 predictor values are valid;
2. a vegetation class is available (`class_id > 0`);
3. a reserve identity is available (`reserve_id > 0`).

The reserve identifier is **not used as a predictor**.

It is used exclusively as a spatial grouping variable for cross-validation.

In [8]:
# ============================================================
# 6. CREATE MACHINE-LEARNING DATASET
# ============================================================

# Valid satellite/environmental predictor values.
valid_predictors_2011 = np.isfinite(
    predictors_2011
).all(axis=0)

# Reference vegetation and reserve must both exist.
valid_labels = (
    (vegetation > 0)
    &
    (reserve > 0)
)

training_mask = (
    valid_predictors_2011
    &
    valid_labels
)


# ------------------------------------------------------------
# X = predictors
# y = vegetation class
# groups = reserve ID
# ------------------------------------------------------------

X = predictors_2011[:, training_mask].T

y = vegetation[training_mask].astype(int)

groups = reserve[training_mask].astype(int)


print("Total raster pixels:", training_mask.size)
print("Training pixels:", training_mask.sum())

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)

print("\nVegetation classes present:")
print(np.unique(y))

print("\nLabelled reserves present:")
print(np.unique(groups))

Total raster pixels: 901888
Training pixels: 24412

X shape: (24412, 12)
y shape: (24412,)
groups shape: (24412,)

Vegetation classes present:
[1 2 3 4 5 6]

Labelled reserves present:
[1 2 3 4 5 6 7]


In [9]:
# ============================================================
# 7. TRAINING DATA AUDIT
# ============================================================

training_table = pd.DataFrame({
    "class_id": y,
    "reserve_id": groups
})


# ------------------------------------------------------------
# Pixels per vegetation class
# ------------------------------------------------------------

class_counts = (
    training_table["class_id"]
    .value_counts()
    .sort_index()
    .rename("n_pixels")
    .to_frame()
)

class_counts["class_name"] = [
    CLASS_NAMES.get(i, f"Class {i}")
    for i in class_counts.index
]

class_counts["area_ha"] = (
    class_counts["n_pixels"] * PIXEL_AREA_HA
)

print("TRAINING PIXELS BY CLASS")
display(class_counts)


# ------------------------------------------------------------
# Pixels per reserve
# ------------------------------------------------------------

reserve_counts = (
    training_table["reserve_id"]
    .value_counts()
    .sort_index()
    .rename("n_pixels")
    .to_frame()
)

reserve_counts["reserve_name"] = [
    RESERVE_NAMES.get(i, f"Reserve {i}")
    for i in reserve_counts.index
]

reserve_counts["area_ha"] = (
    reserve_counts["n_pixels"] * PIXEL_AREA_HA
)

print("\nTRAINING PIXELS BY RESERVE")
display(reserve_counts)


# ------------------------------------------------------------
# Class × reserve table
# ------------------------------------------------------------

class_by_reserve = pd.crosstab(
    training_table["reserve_id"],
    training_table["class_id"]
)

class_by_reserve.index = [
    RESERVE_NAMES.get(i, f"Reserve {i}")
    for i in class_by_reserve.index
]

class_by_reserve.columns = [
    CLASS_NAMES.get(i, f"Class {i}")
    for i in class_by_reserve.columns
]

print("\nCLASS × RESERVE")
display(class_by_reserve)


# Save tables.
class_counts.to_csv(
    OUTPUT_DIR / "training_pixels_by_class.csv"
)

reserve_counts.to_csv(
    OUTPUT_DIR / "training_pixels_by_reserve.csv"
)

class_by_reserve.to_csv(
    OUTPUT_DIR / "class_by_reserve.csv"
)

TRAINING PIXELS BY CLASS


,n_pixels,class_name,area_ha
class_id,,,
1,8908,Eaux libres,801.72
2,7221,Rivages et lieux humides,649.89
3,421,Pelouses et prairies,37.89
4,7391,Forêts,665.19
5,143,Plantations/champs/cultures,12.87
6,328,Milieux construits,29.52



TRAINING PIXELS BY RESERVE


,n_pixels,reserve_name,area_ha
reserve_id,,,
1,3164,Grèves de Cheseaux,284.76
2,3455,Baie d'Yvonand,310.95
3,2650,Cheyres,238.50
4,2908,Grèves de la Corbière,261.72
5,5424,Grèves d'Ostende,488.16
6,4107,Grèves de la Motte,369.63
7,2704,Cudrefin,243.36



CLASS × RESERVE


,Eaux libres,Rivages et lieux humides,Pelouses et prairies,Forêts,Plantations/champs/cultures,Milieux construits
Grèves de Cheseaux,1097,812,76,1074,0,105
Baie d'Yvonand,2249,447,25,712,0,22
Cheyres,773,802,83,944,3,45
Grèves de la Corbière,972,679,10,1202,2,43
Grèves d'Ostende,1987,2236,93,1020,45,43
Grèves de la Motte,966,1567,24,1498,0,52
Cudrefin,864,678,110,941,93,18


## Spatial cross-validation

A random pixel train/test split would be inappropriate because nearby pixels are spatially
autocorrelated and frequently belong to the same vegetation polygons.

Instead, **leave-one-reserve-out cross-validation** is used.

For each fold:

- one entire reserve is excluded;
- the Random Forest is trained using all other labelled reserves;
- the excluded reserve is used as independent test data.

Consequently, each labelled reserve is predicted once by a model that has never seen
training pixels from that reserve.

This directly evaluates the intended application of the model: spatial transfer to a
previously unseen reserve.

In [ ]:
# ============================================================
# 8. LEAVE-ONE-RESERVE-OUT CROSS-VALIDATION
# ============================================================

logo = LeaveOneGroupOut()

all_classes = np.sort(np.unique(y))

oof_predictions = np.full(
    y.shape,
    fill_value=-1,
    dtype=np.int16
)

fold_results = []
pfi_results = []


for fold_number, (train_idx, test_idx) in enumerate(
    logo.split(X, y, groups),
    start=1
):

    held_out_id = int(np.unique(groups[test_idx])[0])

    held_out_name = RESERVE_NAMES.get(
        held_out_id,
        f"Reserve {held_out_id}"
    )

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]


    print("=" * 70)
    print(f"Fold {fold_number}")
    print(f"Held-out reserve: {held_out_id} — {held_out_name}")
    print("Training pixels:", len(train_idx))
    print("Test pixels:", len(test_idx))


    # --------------------------------------------------------
    # Check whether every vegetation class in the full dataset
    # occurs somewhere in the training reserves.
    # --------------------------------------------------------

    classes_in_training = set(np.unique(y_train))

    missing_classes = [
        c for c in all_classes
        if c not in classes_in_training
    ]

    if missing_classes:
        warnings.warn(
            "The following vegetation classes are absent from "
            f"the training data in this fold: {missing_classes}. "
            "The classifier cannot predict classes it has never seen."
        )


    # --------------------------------------------------------
    # Train Random Forest
    # --------------------------------------------------------

    model = RandomForestClassifier(
        **RF_PARAMS
    )

    model.fit(
        X_train,
        y_train
    )


    # --------------------------------------------------------
    # Predict held-out reserve
    # --------------------------------------------------------

    y_pred = model.predict(X_test)

    oof_predictions[test_idx] = y_pred


    # --------------------------------------------------------
    # Performance metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        y_pred
    )

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    kappa = cohen_kappa_score(
        y_test,
        y_pred
    )


    fold_results.append({
        "fold": fold_number,
        "held_out_reserve_id": held_out_id,
        "held_out_reserve": held_out_name,
        "n_train_pixels": len(train_idx),
        "n_test_pixels": len(test_idx),
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "cohen_kappa": kappa
    })


    print(
        f"Accuracy:          {accuracy:.3f}"
    )
    print(
        f"Balanced accuracy: {balanced_accuracy:.3f}"
    )
    print(
        f"Macro F1:          {macro_f1:.3f}"
    )
    print(
        f"Cohen's kappa:     {kappa:.3f}"
    )


    # ========================================================
    # PERMUTATION FEATURE IMPORTANCE
    # ========================================================
    #
    # PFI is calculated ONLY using pixels from the held-out
    # test reserve.
    #
    # This follows the project requirement and prevents the
    # importance calculation from being based on training data.
    # ========================================================

    if len(test_idx) > PFI_MAX_SAMPLES:

        rng = np.random.default_rng(
            RANDOM_STATE + held_out_id
        )

        pfi_local_idx = rng.choice(
            len(test_idx),
            size=PFI_MAX_SAMPLES,
            replace=False
        )

        X_pfi = X_test[pfi_local_idx]
        y_pfi = y_test[pfi_local_idx]

    else:

        X_pfi = X_test
        y_pfi = y_test


    pfi = permutation_importance(
        model,
        X_pfi,
        y_pfi,
        scoring="balanced_accuracy",
        n_repeats=PFI_N_REPEATS,
        random_state=RANDOM_STATE + held_out_id,
        n_jobs=-1
    )


    for feature, mean_imp, std_imp in zip(
        FEATURE_NAMES,
        pfi.importances_mean,
        pfi.importances_std
    ):

        pfi_results.append({
            "fold": fold_number,
            "held_out_reserve_id": held_out_id,
            "held_out_reserve": held_out_name,
            "feature": feature,
            "importance_mean": mean_imp,
            "importance_std_within_fold": std_imp,
            "n_test_pixels_for_pfi": len(y_pfi)
        })


# ------------------------------------------------------------
# Convert results to tables
# ------------------------------------------------------------

cv_results = pd.DataFrame(
    fold_results
)

pfi_by_fold = pd.DataFrame(
    pfi_results
)

print("\nCross-validation completed.")

Fold 1
Held-out reserve: 1 — Grèves de Cheseaux
Training pixels: 21248
Test pixels: 3164
Accuracy:          0.856
Balanced accuracy: 0.547
Macro F1:          0.537
Cohen's kappa:     0.789
Fold 2
Held-out reserve: 2 — Baie d'Yvonand
Training pixels: 20957
Test pixels: 3455


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Accuracy:          0.921
Balanced accuracy: 0.557
Macro F1:          0.464
Cohen's kappa:     0.848


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Fold 3
Held-out reserve: 3 — Cheyres
Training pixels: 21762
Test pixels: 2650
Accuracy:          0.818
Balanced accuracy: 0.448
Macro F1:          0.457
Cohen's kappa:     0.734
Fold 4
Held-out reserve: 4 — Grèves de la Corbière
Training pixels: 21504
Test pixels: 2908
Accuracy:          0.878
Balanced accuracy: 0.452
Macro F1:          0.459
Cohen's kappa:     0.815
Fold 5
Held-out reserve: 5 — Grèves d'Ostende
Training pixels: 18988
Test pixels: 5424
Accuracy:          0.902
Balanced accuracy: 0.540
Macro F1:          0.561
Cohen's kappa:     0.851
Fold 6
Held-out reserve: 6 — Grèves de la Motte
Training pixels: 20305
Test pixels: 4107


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Accuracy:          0.864
Balanced accuracy: 0.533
Macro F1:          0.441
Cohen's kappa:     0.795


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Fold 7
Held-out reserve: 7 — Cudrefin
Training pixels: 21708
Test pixels: 2704
Accuracy:          0.889
Balanced accuracy: 0.581
Macro F1:          0.595
Cohen's kappa:     0.841


In [ ]:
# ============================================================
# 9. CROSS-VALIDATION RESULTS
# ============================================================

display(
    cv_results.round(3)
)

cv_results.to_csv(
    OUTPUT_DIR / "cv_metrics_by_reserve.csv",
    index=False
)


print("\nMEAN PERFORMANCE ACROSS RESERVES")

summary_metrics = (
    cv_results[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "cohen_kappa"
        ]
    ]
    .agg(["mean", "std"])
    .T
)

display(
    summary_metrics.round(3)
)

summary_metrics.to_csv(
    OUTPUT_DIR / "cv_metrics_summary.csv"
)

In [ ]:
# ============================================================
# 10. OUT-OF-FOLD PERFORMANCE
# ============================================================
#
# Every labelled pixel is represented exactly once here:
# when its entire reserve was held out.
# ============================================================

assert np.all(oof_predictions >= 0), (
    "Some training pixels did not receive an out-of-fold prediction."
)


oof_accuracy = accuracy_score(
    y,
    oof_predictions
)

oof_balanced_accuracy = balanced_accuracy_score(
    y,
    oof_predictions
)

oof_macro_f1 = f1_score(
    y,
    oof_predictions,
    average="macro",
    zero_division=0
)

oof_kappa = cohen_kappa_score(
    y,
    oof_predictions
)


print("OUT-OF-FOLD PERFORMANCE")
print("-----------------------")
print(f"Accuracy:          {oof_accuracy:.3f}")
print(f"Balanced accuracy: {oof_balanced_accuracy:.3f}")
print(f"Macro F1:          {oof_macro_f1:.3f}")
print(f"Cohen's kappa:     {oof_kappa:.3f}")

In [ ]:
# ============================================================
# 11. PER-CLASS PERFORMANCE
# ============================================================

class_labels = [
    CLASS_NAMES.get(c, f"Class {c}")
    for c in all_classes
]

report = classification_report(
    y,
    oof_predictions,
    labels=all_classes,
    target_names=class_labels,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report
).T

display(
    report_df.round(3)
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report_oof.csv"
)

In [ ]:
# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y,
    oof_predictions,
    labels=all_classes
)

cm_df = pd.DataFrame(
    cm,
    index=class_labels,
    columns=class_labels
)

display(cm_df)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix_oof.csv"
)


# ------------------------------------------------------------
# Normalised confusion matrix figure
# ------------------------------------------------------------

cm_normalised = confusion_matrix(
    y,
    oof_predictions,
    labels=all_classes,
    normalize="true"
)

fig, ax = plt.subplots(
    figsize=(10, 8)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_normalised,
    display_labels=class_labels
)

disp.plot(
    ax=ax,
    values_format=".2f",
    colorbar=False,
    xticks_rotation=45
)

ax.set_title(
    "Leave-one-reserve-out confusion matrix\n"
    "(normalised by reference class)"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "confusion_matrix_oof.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Permutation feature importance

Permutation feature importance measures the decrease in predictive performance after the
values of one predictor are randomly permuted.

A useful predictor should cause model performance to decrease when its values are
destroyed.

PFI is calculated separately for each **held-out reserve**, using balanced accuracy as the
performance metric.

The mean importance across reserves is reported below. This provides a spatially
independent estimate of which environmental variables contribute most strongly to
vegetation classification.

In [ ]:
# ============================================================
# 13. PERMUTATION FEATURE IMPORTANCE SUMMARY
# ============================================================

pfi_by_fold.to_csv(
    OUTPUT_DIR / "permutation_importance_by_fold.csv",
    index=False
)


pfi_summary = (
    pfi_by_fold
    .groupby("feature")["importance_mean"]
    .agg(
        mean_importance="mean",
        std_between_reserves="std",
        min_importance="min",
        max_importance="max"
    )
    .sort_values(
        "mean_importance",
        ascending=False
    )
)

display(
    pfi_summary.round(4)
)

pfi_summary.to_csv(
    OUTPUT_DIR / "permutation_importance_summary.csv"
)

In [ ]:
# ============================================================
# 14. PFI FIGURE
# ============================================================

plot_df = (
    pfi_summary
    .sort_values(
        "mean_importance",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.barh(
    plot_df.index,
    plot_df["mean_importance"]
)

ax.errorbar(
    plot_df["mean_importance"],
    plot_df.index,
    xerr=plot_df["std_between_reserves"],
    fmt="none",
    capsize=3
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_xlabel(
    "Decrease in balanced accuracy after permutation"
)

ax.set_ylabel(
    "Predictor"
)

ax.set_title(
    "Permutation feature importance\n"
    "Mean ± SD across held-out reserves"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "permutation_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Final model

Cross-validation models are used only to estimate generalisation performance.

After validation, a new **final Random Forest** is trained using all available labelled
reserves.

This maximises the amount of reference information available when predicting:

- vegetation in areas lacking 2011 labels;
- and the equivalent predictor stack for 2025.

Cross-validation performance remains the estimate of expected prediction accuracy.
Training performance of the final model is not used as validation.

In [ ]:
# ============================================================
# 15. TRAIN FINAL RANDOM FOREST
# ============================================================

final_model = RandomForestClassifier(
    **RF_PARAMS
)

final_model.fit(
    X,
    y
)

print(
    "Final Random Forest trained on",
    len(y),
    "labelled pixels from",
    len(np.unique(groups)),
    "reserves."
)


# Save model together with metadata.

model_package = {
    "model": final_model,
    "feature_names": FEATURE_NAMES,
    "class_names": CLASS_NAMES,
    "reserve_names": RESERVE_NAMES,
    "rf_parameters": RF_PARAMS
}

joblib.dump(
    model_package,
    OUTPUT_DIR / "random_forest_final.joblib"
)

print(
    "Saved:",
    OUTPUT_DIR / "random_forest_final.joblib"
)

In [ ]:
# ============================================================
# 16. FUNCTIONS FOR PREDICTING COMPLETE RASTERS
# ============================================================

def predict_stack(
    model,
    predictor_stack,
    chunk_size=100000
):
    """
    Predict vegetation class and maximum Random Forest class
    probability for every valid raster pixel.

    Prediction is performed in chunks to avoid excessive memory use.
    """

    n_features, rows, cols = predictor_stack.shape

    flat = (
        predictor_stack
        .reshape(n_features, -1)
        .T
    )

    valid = np.isfinite(
        flat
    ).all(axis=1)

    valid_indices = np.flatnonzero(
        valid
    )

    predicted_class = np.zeros(
        flat.shape[0],
        dtype=np.uint8
    )

    confidence = np.full(
        flat.shape[0],
        np.nan,
        dtype=np.float32
    )


    for start in range(
        0,
        len(valid_indices),
        chunk_size
    ):

        batch_indices = valid_indices[
            start:start + chunk_size
        ]

        X_batch = flat[
            batch_indices
        ]

        probabilities = model.predict_proba(
            X_batch
        )

        best_probability_index = np.argmax(
            probabilities,
            axis=1
        )

        batch_classes = model.classes_[
            best_probability_index
        ]

        batch_confidence = probabilities[
            np.arange(len(batch_indices)),
            best_probability_index
        ]

        predicted_class[
            batch_indices
        ] = batch_classes.astype(
            np.uint8
        )

        confidence[
            batch_indices
        ] = batch_confidence.astype(
            np.float32
        )


    predicted_class = predicted_class.reshape(
        rows,
        cols
    )

    confidence = confidence.reshape(
        rows,
        cols
    )

    return predicted_class, confidence


def save_class_raster(
    array,
    path,
    reference_profile
):
    """
    Save categorical vegetation / status raster.
    Value 0 is NoData.
    """

    profile = reference_profile.copy()

    profile.update(
        count=1,
        dtype="uint8",
        nodata=0,
        compress="lzw"
    )

    with rasterio.open(
        path,
        "w",
        **profile
    ) as dst:

        dst.write(
            array.astype(np.uint8),
            1
        )


def save_float_raster(
    array,
    path,
    reference_profile
):
    """
    Save continuous float raster.
    """

    nodata_value = -9999.0

    output = np.where(
        np.isfinite(array),
        array,
        nodata_value
    ).astype(np.float32)

    profile = reference_profile.copy()

    profile.update(
        count=1,
        dtype="float32",
        nodata=nodata_value,
        compress="lzw"
    )

    with rasterio.open(
        path,
        "w",
        **profile
    ) as dst:

        dst.write(
            output,
            1
        )

In [ ]:
# ============================================================
# 18. SAVE VEGETATION PREDICTIONS
# ============================================================

save_class_raster(
    predicted_2011,
    OUTPUT_DIR / "vegetation_predicted_2011_RF.tif",
    meta_2011["profile"]
)

save_class_raster(
    predicted_2025,
    OUTPUT_DIR / "vegetation_predicted_2025_RF.tif",
    meta_2025["profile"]
)

save_float_raster(
    confidence_2011,
    OUTPUT_DIR / "prediction_confidence_2011.tif",
    meta_2011["profile"]
)

save_float_raster(
    confidence_2025,
    OUTPUT_DIR / "prediction_confidence_2025.tif",
    meta_2025["profile"]
)


print("Prediction rasters saved.")

## 2011 baseline map

For 2011, the original vegetation map is preferable to a model prediction wherever a
reference class exists.

A hybrid baseline is therefore produced:

- **reference vegetation class** where the 2010–2011 vegetation map contains a class;
- **Random Forest prediction** only where the reference map is missing.

This avoids replacing known reference information with an in-sample model prediction.

In [ ]:
# ============================================================
# 19. 2011 REFERENCE + MODEL BASELINE
# ============================================================

baseline_2011 = predicted_2011.copy()

reference_mask = vegetation > 0

# Where actual 2011 vegetation information exists,
# use the reference map rather than model prediction.
baseline_2011[
    reference_mask
] = vegetation[
    reference_mask
].astype(np.uint8)


# ------------------------------------------------------------
# Identify where the 2011 baseline comes from
#
# 0 = no data
# 1 = 2010–2011 reference vegetation map
# 2 = Random Forest prediction
# ------------------------------------------------------------

baseline_source = np.zeros(
    vegetation.shape,
    dtype=np.uint8
)

baseline_source[
    reference_mask
] = 1

model_only_mask = (
    (~reference_mask)
    &
    (predicted_2011 > 0)
)

baseline_source[
    model_only_mask
] = 2


save_class_raster(
    baseline_2011,
    OUTPUT_DIR / "vegetation_baseline_2011_reference_plus_RF.tif",
    meta_2011["profile"]
)

save_class_raster(
    baseline_source,
    OUTPUT_DIR / "vegetation_baseline_2011_source.tif",
    meta_2011["profile"]
)

print("2011 baseline saved.")

In [ ]:
# ============================================================
# 20. CANDIDATE VEGETATION CHANGE
# ============================================================

valid_change = (
    (baseline_2011 > 0)
    &
    (predicted_2025 > 0)
)


changed = (
    valid_change
    &
    (baseline_2011 != predicted_2025)
)


# ------------------------------------------------------------
# Confidence associated with temporal comparison
# ------------------------------------------------------------

change_confidence = confidence_2025.copy()

# If 2011 itself was modelled rather than observed,
# require confidence in BOTH dates.

modelled_2011 = (
    baseline_source == 2
)

change_confidence[
    modelled_2011
] = np.minimum(
    confidence_2011[modelled_2011],
    confidence_2025[modelled_2011]
)


# ------------------------------------------------------------
# Change status
#
# 0 = NoData
# 1 = Same vegetation class
# 2 = Candidate change, confidence < threshold
# 3 = Candidate change, confidence >= threshold
# ------------------------------------------------------------

change_status = np.zeros(
    vegetation.shape,
    dtype=np.uint8
)

stable = (
    valid_change
    &
    (~changed)
)

low_confidence_change = (
    changed
    &
    (
        change_confidence
        <
        CHANGE_CONFIDENCE_THRESHOLD
    )
)

high_confidence_change = (
    changed
    &
    (
        change_confidence
        >=
        CHANGE_CONFIDENCE_THRESHOLD
    )
)


change_status[
    stable
] = 1

change_status[
    low_confidence_change
] = 2

change_status[
    high_confidence_change
] = 3


save_class_raster(
    change_status,
    OUTPUT_DIR / "candidate_change_2011_2025.tif",
    meta_2011["profile"]
)

save_float_raster(
    change_confidence,
    OUTPUT_DIR / "change_comparison_confidence.tif",
    meta_2011["profile"]
)


print("Candidate change raster saved.")

In [ ]:
# ============================================================
# 21. VEGETATION TRANSITION MATRIX
# ============================================================

from_values = baseline_2011[
    valid_change
]

to_values = predicted_2025[
    valid_change
]


transition_pixels = pd.crosstab(
    pd.Series(
        from_values,
        name="2011"
    ),
    pd.Series(
        to_values,
        name="2025"
    )
)


# Ensure all classes are represented.
transition_pixels = transition_pixels.reindex(
    index=all_classes,
    columns=all_classes,
    fill_value=0
)


transition_pixels.index = [
    CLASS_NAMES.get(i, f"Class {i}")
    for i in transition_pixels.index
]

transition_pixels.columns = [
    CLASS_NAMES.get(i, f"Class {i}")
    for i in transition_pixels.columns
]


transition_area_ha = (
    transition_pixels
    *
    PIXEL_AREA_HA
)


print("TRANSITION MATRIX — PIXELS")
display(transition_pixels)

print("\nTRANSITION MATRIX — HECTARES")
display(
    transition_area_ha.round(2)
)


transition_pixels.to_csv(
    OUTPUT_DIR / "transition_matrix_pixels.csv"
)

transition_area_ha.to_csv(
    OUTPUT_DIR / "transition_matrix_hectares.csv"
)

In [ ]:
# ============================================================
# 22. SUMMARY OF CANDIDATE CHANGE
# ============================================================

n_valid = np.sum(
    valid_change
)

n_stable = np.sum(
    stable
)

n_changed = np.sum(
    changed
)

n_high_conf_change = np.sum(
    high_confidence_change
)

n_low_conf_change = np.sum(
    low_confidence_change
)


change_summary = pd.DataFrame({
    "category": [
        "Valid comparison area",
        "Same mapped class",
        "All candidate changes",
        "High-confidence candidate changes",
        "Lower-confidence candidate changes"
    ],

    "pixels": [
        n_valid,
        n_stable,
        n_changed,
        n_high_conf_change,
        n_low_conf_change
    ]
})


change_summary["area_ha"] = (
    change_summary["pixels"]
    *
    PIXEL_AREA_HA
)


display(
    change_summary
)


if n_valid > 0:

    print(
        "\nFraction of valid pixels with different mapped class:",
        f"{100 * n_changed / n_valid:.1f}%"
    )

    print(
        "Fraction with high-confidence candidate change:",
        f"{100 * n_high_conf_change / n_valid:.1f}%"
    )


change_summary.to_csv(
    OUTPUT_DIR / "candidate_change_summary.csv",
    index=False
)


print(
    "\nIMPORTANT: the temporal change percentage must not "
    "be interpreted as the inverse of model accuracy. "
    "Classification error and ecological change are different "
    "sources of disagreement."
)

In [ ]:
# ============================================================
# 23. PREDICTOR DISTRIBUTION DIAGNOSTICS
# ============================================================

def sample_predictors(
    stack,
    max_pixels=100000,
    random_state=42
):

    n_features = stack.shape[0]

    flat = (
        stack
        .reshape(n_features, -1)
        .T
    )

    valid = np.isfinite(
        flat
    ).all(axis=1)

    valid_idx = np.flatnonzero(
        valid
    )

    if len(valid_idx) > max_pixels:

        rng = np.random.default_rng(
            random_state
        )

        valid_idx = rng.choice(
            valid_idx,
            size=max_pixels,
            replace=False
        )

    return flat[
        valid_idx
    ]


sample_2011 = sample_predictors(
    predictors_2011,
    random_state=RANDOM_STATE
)

sample_2025 = sample_predictors(
    predictors_2025,
    random_state=RANDOM_STATE + 1
)


diagnostic_rows = []

for j, feature in enumerate(
    FEATURE_NAMES
):

    q2011 = np.percentile(
        sample_2011[:, j],
        [5, 50, 95]
    )

    q2025 = np.percentile(
        sample_2025[:, j],
        [5, 50, 95]
    )

    diagnostic_rows.append({
        "feature": feature,

        "2011_q05": q2011[0],
        "2011_median": q2011[1],
        "2011_q95": q2011[2],

        "2025_q05": q2025[0],
        "2025_median": q2025[1],
        "2025_q95": q2025[2],

        "median_difference_2025_minus_2011":
            q2025[1] - q2011[1]
    })


predictor_diagnostics = pd.DataFrame(
    diagnostic_rows
)


display(
    predictor_diagnostics.round(4)
)


predictor_diagnostics.to_csv(
    OUTPUT_DIR /
    "predictor_distribution_2011_vs_2025.csv",
    index=False
)

In [ ]:
# ============================================================
# 24. PROJECT OUTPUTS
# ============================================================

print("OUTPUT TABLES / MODELS")
print("----------------------")

for path in sorted(
    OUTPUT_DIR.iterdir()
):
    print(path.name)


print("\nFIGURES")
print("-------")

for path in sorted(
    FIGURE_DIR.iterdir()
):
    print(path.name)